# Dataset Overview (From Kaggle)

Centrifugal chillers are critical components in modern HVAC systems, consuming significant electrical energy in large commercial and industrial buildings. Early and accurate Fault Detection and Diagnosis (FDD) is essential for maintaining energy efficiency, preventing unexpected shutdowns, and reducing maintenance costs.

This dataset provides preprocessed telemetry sensor data optimized for developing, benchmarking, and evaluating machine learning and deep learning models for time-series classification and chiller fault diagnosis.

## Data Source (From Kaggle)

The dataset is derived from the benchmark ASHRAE Research Project 1043 (RP-1043), which contains experimental measurements collected under controlled conditions from a 90-ton water-cooled centrifugal chiller operating across various operating loads and fault severity levels.

## Data Collection & Physical Fault Injection (From Kaggle)

Unlike purely synthetic datasets generated via mathematical noise, this dataset originates from empirical experiments conducted on an actual industrial-grade **90-ton water-cooled centrifugal chiller testbed** under ASHRAE Research Project 1043.

The data generation adheres to rigorous physical measurement protocols:

- **Real Operational Testbed:** Run under dynamic, real-world cooling loads and condenser water conditions to capture genuine operational transients and system inertia.
- **Controlled Physical Fault Injection:** Faults were deliberately and physically introduced into the operating system rather than simulated via heuristic rules. For example:
  - _Refrigerant leaks and overcharges_ were realized by physically removing or adding controlled mass increments of refrigerant.
  - _Condenser fouling_ was simulated by physically restricting tube channels to emulate scale/deposit resistance.
  - _Non-condensables_ were induced by bleeding measured amounts of nitrogen/air into the circuit.
  - _Flow reductions and oil imbalances_ were directly actuated via throttling valves and precision oil injection/extraction.
- **Graded Fault Severity:** Each physical fault was evaluated across four progressive severity levels (SL1 to SL4) alongside varying thermal load ratios, capturing both subtle micro-deviations at early stages and acute system degradation.

## Fault Classes & Labels (From Kaggle)

The dataset covers normal operation along with several distinct typical chiller fault types, labeled by file name:

- **normal**: Normal baseline operation under standard operating conditions.
- **cf** _(Condenser Fouling)_: Accumulation of deposits or scaling on the condenser water tube surfaces, impairing heat transfer.
- **eo** _(Excess Oil)_: Overcharge of lubrication oil into the refrigerant circuit, degrading cooling performance.
- **fwc** _(Reduced Condenser Water Flow)_: Decreased water flow through the condenser due to valve or pump anomalies.
- **fwe** _(Reduced Evaporator Water Flow)_: Restricted chilled water flow across the evaporator.
- **nc** _(Non-condensable Gas)_: Presence of non-condensable gases (such as air) trapped inside the refrigerant circuit.
- **rl** _(Refrigerant Leak / Undercharge)_: Deficit in system refrigerant charge, reducing cooling capacity.
- **ro** _(Refrigerant Overcharge)_: Excess refrigerant charge, leading to elevated condenser pressure and compressor stress.

## Imports

In [5]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

## Visualization aesthetic configuration

In [6]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

FAULT_ORDER = ['normal', 'cf', 'fwe', 'fwc', 'ro', 'rl', 'nc', 'eo']
FAULT_NAMES = {
    'normal': 'Normal Operation',
    'cf': 'Condenser Fouling (CF)',
    'fwe': 'Reduced Evaporator Flow (FWE)',
    'fwc': 'Reduced Condenser Flow (FWC)',
    'ro': 'Refrigerant Overcharge (RO)',
    'rl': 'Refrigerant Leakage (RL)',
    'nc': 'Non-condensable Gas (NC)',
    'eo': 'Excess Oil (EO)'
}

PALETTE = {
    'normal': '#2ecc71',  
    'cf':     '#e67e22',  
    'fwe':    '#3498db',  
    'fwc':    '#9b59b6',  
    'ro':     '#e74c3c',  
    'rl':     '#1abc9c', 
    'nc':     '#f39c12',  
    'eo':     '#34495e'   
}

# Resolve data directory

In [7]:
BASE_DIR = Path.cwd()
if (BASE_DIR / "data").exists():
    DATA_DIR = BASE_DIR / "data"
elif (BASE_DIR.parent / "data").exists():
    DATA_DIR = BASE_DIR.parent / "data"

print(f"Data directory: {DATA_DIR}")

Data directory: /home/henriquias/projects/eda-chiller-fault-detection-diagnostics-dataset/data


##  Read and Stratified Sample data from 8 parquet files

In [8]:
samples = []
N_PER_FAULT = 10000

for fault_name in FAULT_ORDER:
    fpath = DATA_DIR / f"{fault_name}.parquet"
    if not fpath.exists():
        print(f"Warning: File {fpath} not found!")
        continue
    
    t = pq.read_table(fpath)
    df_temp = t.to_pandas()
    
    if fault_name == 'normal':
        sampled = df_temp.sample(n=min(len(df_temp), N_PER_FAULT), random_state=42)
    else:
        n_per_sev = N_PER_FAULT // 4
        sampled = df_temp.groupby('severity_level', group_keys=False).apply(
            lambda g: g.sample(n=min(len(g), n_per_sev), random_state=42)
        )
    samples.append(sampled)
    print(f"✓ Loaded {fault_name:<8}: {len(sampled):,d} rows sampled from {len(df_temp):,d} total records.")

df = pd.concat(samples, ignore_index=True)
df['fault'] = pd.Categorical(df['fault'], categories=FAULT_ORDER, ordered=True)
print(f"\nTotal sampled DataFrame shape: {len(df):,d} rows, {df.shape[1]} columns.")

✓ Loaded normal  : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded cf      : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded fwe     : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded fwc     : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded ro      : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded rl      : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded nc      : 10,000 rows sampled from 2,000,000 total records.
✓ Loaded eo      : 10,000 rows sampled from 2,000,000 total records.

Total sampled DataFrame shape: 80,000 rows, 24 columns.
